In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from datetime import datetime , timedelta

In [ ]:
API_KEY = '6RvcXqm7yDGUlYuJZFMrFseHTVDAZGzJmmuT9zLl'

start_date = datetime(2015,1,1)
end_date = datetime(2026,5,30)

all_asteroids = []

current = start_date

while current < end_date:
  next_date = current + timedelta(days=7)

  params = {
      'start_date': current.strftime('%Y-%m-%d'),
      'end_date': next_date.strftime('%Y-%m-%d'),
      'api_key' : API_KEY
  }
  response = requests.get(
      "https://api.nasa.gov/neo/rest/v1/feed",
      params=params
  )
  data = response.json()

  for date in data['near_earth_objects']:
      all_asteroids.extend(data['near_earth_objects'][date])


  current = next_date
  print(f'Done : {current.strftime('%Y-%m-%d')}')





Done : 2015-01-08
Done : 2015-01-15
Done : 2015-01-22
Done : 2015-01-29
Done : 2015-02-05
Done : 2015-02-12
Done : 2015-02-19
Done : 2015-02-26
Done : 2015-03-05
Done : 2015-03-12
Done : 2015-03-19
Done : 2015-03-26
Done : 2015-04-02
Done : 2015-04-09
Done : 2015-04-16
Done : 2015-04-23
Done : 2015-04-30
Done : 2015-05-07
Done : 2015-05-14
Done : 2015-05-21
Done : 2015-05-28
Done : 2015-06-04
Done : 2015-06-11
Done : 2015-06-18
Done : 2015-06-25
Done : 2015-07-02
Done : 2015-07-09
Done : 2015-07-16
Done : 2015-07-23
Done : 2015-07-30
Done : 2015-08-06
Done : 2015-08-13
Done : 2015-08-20
Done : 2015-08-27
Done : 2015-09-03
Done : 2015-09-10
Done : 2015-09-17
Done : 2015-09-24
Done : 2015-10-01
Done : 2015-10-08
Done : 2015-10-15
Done : 2015-10-22
Done : 2015-10-29
Done : 2015-11-05
Done : 2015-11-12
Done : 2015-11-19
Done : 2015-11-26
Done : 2015-12-03
Done : 2015-12-10
Done : 2015-12-17
Done : 2015-12-24
Done : 2015-12-31
Done : 2016-01-07
Done : 2016-01-14
Done : 2016-01-21
Done : 201

In [ ]:
df = pd.DataFrame(all_asteroids)
print(df.shape)

df.to_csv("asteroid_final.csv", index=False)

from google.colab import drive
drive.mount('/content/drive')
df.to_csv("/content/drive/MyDrive/asteroid_final.csv", index=False)
print("Drive pe save ho gaya!")

(38573, 11)
Mounted at /content/drive
Drive pe save ho gaya!


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/asteroid_final.csv")

In [ ]:
df.head()

,links,id,neo_reference_id,name,nasa_jpl_url,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid,close_approach_data,is_sentry_object,sentry_data
0,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,2085990,2085990,85990 (1999 JV6),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,20.27,{'kilometers': {'estimated_diameter_min': 0.23...,True,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN
1,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,2523915,2523915,523915 (1997 VM4),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,18.57,{'kilometers': {'estimated_diameter_min': 0.51...,False,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN
2,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3263533,3263533,(2004 XG29),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,25.66,{'kilometers': {'estimated_diameter_min': 0.01...,False,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN
3,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3426635,3426635,(2008 RU),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,22.40,{'kilometers': {'estimated_diameter_min': 0.08...,False,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN
4,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3601924,3601924,(2012 FO52),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,22.63,{'kilometers': {'estimated_diameter_min': 0.07...,False,"[{'close_approach_date': '2015-01-05', 'close_...",False,NaN


In [ ]:
df['is_potentially_hazardous_asteroid'].value_counts()

,count
is_potentially_hazardous_asteroid,
False,35170
True,3403


In [ ]:
import ast

df['estimated_diameter'] = df['estimated_diameter'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df['close_approach_data'] = df['close_approach_data'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [ ]:
df['min_diameter'] = df['estimated_diameter'].apply(
    lambda x : x['kilometers']['estimated_diameter_min']
)

df['max_diameter'] = df['estimated_diameter'].apply(
    lambda x : x['kilometers']['estimated_diameter_max']
)

df['miss_dis_km'] = df['close_approach_data'].apply(
    lambda x : float(x[0]['miss_distance']['kilometers'])
    if len(x) > 0 else None
)

df['velocity_km_s'] = df['close_approach_data'].apply(
    lambda x : float(x[0]['relative_velocity']['kilometers_per_second'])
    if len(x) > 0 else None
)

df['diameter_avg'] = (df['min_diameter'] + df['max_diameter']) / 2

df['threat_score'] = df['velocity_km_s'] / df['miss_dis_km']

df['size_velocity'] = df['diameter_avg'] * df['velocity_km_s']

In [ ]:
df.isnull().sum()

,0
links,0
id,0
neo_reference_id,0
name,0
nasa_jpl_url,0
absolute_magnitude_h,0
estimated_diameter,0
is_potentially_hazardous_asteroid,0
close_approach_data,0
is_sentry_object,0


In [ ]:
df.drop(columns=['links', 'nasa_jpl_url',
                 'neo_reference_id',
                 'estimated_diameter',
                 'close_approach_data',
                 'sentry_data'], inplace=True)

print(df.shape)
print(df.columns)

(38573, 12)
Index(['id', 'name', 'absolute_magnitude_h',
       'is_potentially_hazardous_asteroid', 'is_sentry_object', 'min_diameter',
       'max_diameter', 'miss_dis_km', 'velocity_km_s', 'diameter_avg',
       'threat_score', 'size_velocity'],
      dtype='object')


In [ ]:
df.isnull().sum().sum()

np.int64(0)

In [ ]:
df['is_sentry_object'].value_counts()

,count
is_sentry_object,
False,36696
True,1877


In [ ]:
df.drop(columns=['is_sentry_object', 'id', 'name'], inplace=True)
print(df.columns)

Index(['absolute_magnitude_h', 'is_potentially_hazardous_asteroid',
       'min_diameter', 'max_diameter', 'miss_dis_km', 'velocity_km_s',
       'diameter_avg', 'threat_score', 'size_velocity'],
      dtype='object')


In [ ]:
API_KEY = '6RvcXqm7yDGUlYuJZFMrFseHTVDAZGzJmmuT9zLl'
url = 'https://api.nasa.gov/DONKI/FLR'
params = {
      'start_date': current.strftime('%Y-%m-%d'),
      'end_date': next_date.strftime('%Y-%m-%d'),
      'api_key' : API_KEY
  }

response = requests.get(url,params=params)

solar_df = pd.DataFrame(response.json())
print(solar_df.shape)
print(solar_df.head())

(16, 15)
                         flrID      catalog  \
0  2026-05-02T10:44:00-FLR-001  M2M_CATALOG   
1  2026-05-02T18:30:00-FLR-001  M2M_CATALOG   
2  2026-05-04T01:13:00-FLR-001  M2M_CATALOG   
3  2026-05-07T14:20:00-FLR-001  M2M_CATALOG   
4  2026-05-10T13:19:00-FLR-001  M2M_CATALOG   

                                 instruments          beginTime  \
0  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-02T10:44Z   
1  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-02T18:30Z   
2  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-04T01:13Z   
3  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-07T14:20Z   
4  [{'displayName': 'GOES-P: EXIS 1.0-8.0'}]  2026-05-10T13:19Z   

            peakTime            endTime classType sourceLocation  \
0  2026-05-02T11:04Z  2026-05-02T11:26Z      C4.0         N16W75   
1  2026-05-02T18:38Z  2026-05-02T18:42Z      C8.7         N16W90   
2  2026-05-04T01:33Z  2026-05-04T01:44Z      M1.8         N17W80   
3  2026-05-07T15:14Z  2026-05-07T

In [ ]:
solar_df = solar_df[['flrID', 'beginTime',
                      'classType', 'sourceLocation',
                      'activeRegionNum']]

print(solar_df.shape)
print(solar_df['classType'].value_counts())

(16, 5)
classType
M1.9    2
C8.7    1
C4.0    1
M2.6    1
M5.7    1
C1.4    1
M1.8    1
C3.3    1
C6.7    1
C9.5    1
M1.3    1
M1.4    1
B8.2    1
M2.3    1
M1.1    1
Name: count, dtype: int64


In [ ]:
def flare_risk(classType):
  if classType.startswith('X'):
    return 3
  elif classType.startswith('M'):
    return 2
  elif classType.startswith('C'):
    return 1
  else :
    return 0


solar_df['flare_risk'] = solar_df['classType'].apply(flare_risk)
print(solar_df[['classType', 'flare_risk']])

   classType  flare_risk
0       C4.0           1
1       C8.7           1
2       M1.8           2
3       M2.6           2
4       M5.7           2
5       C1.4           1
6       C3.3           1
7       C6.7           1
8       C9.5           1
9       M1.9           2
10      M1.3           2
11      M1.9           2
12      M1.4           2
13      B8.2           0
14      M2.3           2
15      M1.1           2


In [ ]:
avg_flare_risk = solar_df['flare_risk'].mean()
print(f'Average flare risk is {avg_flare_risk}')

Average flare risk is 1.5


In [ ]:
df['flare_risk'] = avg_flare_risk
print(df.shape)
print(df.head())

(38573, 10)
   absolute_magnitude_h  is_potentially_hazardous_asteroid  min_diameter  \
0                 20.27                               True      0.234723   
1                 18.57                              False      0.513517   
2                 25.66                              False      0.019613   
3                 22.40                              False      0.088015   
4                 22.63                              False      0.079169   

   max_diameter   miss_dis_km  velocity_km_s  diameter_avg  threat_score  \
0      0.524856  1.246329e+07       7.694743      0.379789  6.173925e-07   
1      1.148259  5.815821e+07      16.169622      0.830888  2.780282e-07   
2      0.043857  3.492914e+07      12.530123      0.031735  3.587298e-07   
3      0.196807  2.459521e+07      12.595836      0.142411  5.121255e-07   
4      0.177027  5.554271e+07      15.775064      0.128098  2.840168e-07   

   size_velocity  flare_risk  
0       2.922380         1.5  
1      13.43

In [ ]:
url = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync'

params = {
    'query' : 'select pl_name , pl_orbsmax , pl_rade , pl_bmasse, st_teff, sy_dist from pscomppars where pl_orbsmax is not null',
    'format' : 'json'
}
response = requests.get(url, params=params)
# print(response.status_code)
# print(response.text[:500])
exo_df = pd.DataFrame(response.json())
print(exo_df.shape)
print(exo_df.head())


(5866, 6)
         pl_name  pl_orbsmax   pl_rade  pl_bmasse  st_teff   sy_dist
0  Kepler-1167 b     0.01750  1.710000      3.570   4971.0   820.905
1  Kepler-1740 b     0.07790  3.323214     11.000   5705.0  1061.770
2  Kepler-1581 b     0.06865  0.800000      0.437   6022.0   493.175
3   Kepler-644 b     0.04641  3.150000     10.100   6747.0  1318.050
4  Kepler-1752 b     0.26980  4.540605     18.700   5446.0   962.888


In [ ]:
def habitable_score(row):
  score=0

  if 4000<= row['st_teff'] <= 7000:
    score =+ 1

  if 0.5< row['pl_orbsmax']<= 2.0:
    score+=1

  if 0.5<= row['pl_rade']:
    score += 1

  return score


exo_df['habitable_score'] = exo_df.apply(habitable_score , axis=1)
print(exo_df['habitable_score'].value_counts())

habitable_score
2    4555
1     855
3     452
0       4
Name: count, dtype: int64


In [ ]:
avg_habitable = exo_df['habitable_score'].mean()
print(f"Average Habitability Score: {avg_habitable}")

Average Habitability Score: 1.9299352199113535


In [ ]:
df['habilability_score'] = avg_habitable

print(df.shape)
print(df.columns.tolist())

(38573, 11)
['absolute_magnitude_h', 'is_potentially_hazardous_asteroid', 'min_diameter', 'max_diameter', 'miss_dis_km', 'velocity_km_s', 'diameter_avg', 'threat_score', 'size_velocity', 'flare_risk', 'habilability_score']


In [ ]:
X = df[['absolute_magnitude_h', 'min_diameter', 'max_diameter', 'miss_dis_km', 'velocity_km_s', 'flare_risk', 'habilability_score', 'diameter_avg', 'threat_score', 'size_velocity']]
y = df['is_potentially_hazardous_asteroid'].astype(int)

In [ ]:
print('Befor SMOTE : ' , y.value_counts())

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

Xtrain , Xtest , ytrain , ytest = train_test_split(X, y , random_state=42 , test_size=0.2 , stratify = y)

smote = SMOTE(random_state=42)

Xtrain_smote , ytrain_smote = smote.fit_resample(Xtrain, ytrain)

print("After SMOTE:", pd.Series(ytrain_smote).value_counts())
print("X_train shape:", Xtrain_smote.shape)

Befor SMOTE :  is_potentially_hazardous_asteroid
0    35170
1     3403
Name: count, dtype: int64
After SMOTE: is_potentially_hazardous_asteroid
0    28136
1    28136
Name: count, dtype: int64
X_train shape: (56272, 10)


In [ ]:
from xgboost import XGBClassifier
xgbmodel = XGBClassifier(n_estimators=500 , max_depth=5, learning_rate=0.05 , colsample_bytree= 0.8 , random_state=42 , eval_metric='logloss')
xgbmodel.fit(Xtrain_smote , ytrain_smote)

from sklearn.metrics import accuracy_score , f1_score , precision_score, recall_score

ypred = xgbmodel.predict(Xtest)

print('Accuracy score is ' , accuracy_score(ytest, ypred))
print('Precision score is ' , precision_score(ytest, ypred))
print('F1 score is ' , f1_score(ytest, ypred))
print('Recall score is ' , recall_score(ytest, ypred))

Accuracy score is  0.8528839922229423
Precision score is  0.36615566037735847
F1 score is  0.5225073622212874
Recall score is  0.9118942731277533


In [ ]:
yprobab = xgbmodel.predict_proba(Xtest)[:, 1]

for threshold in [0.3,0.4,0.5,0.6,0.7,0.8]:
  ypred_t = (yprobab >= threshold).astype(int)
  print(f"Threshold {threshold}:")
  print('Accuracy score is ' , accuracy_score(ytest, ypred_t))
  print('Precision score is ' , precision_score(ytest, ypred))
  print('F1 score is ' , f1_score(ytest, ypred))
  print('Recall score is  ' , recall_score(ytest, ypred))
  print()

Threshold 0.3:
Accuracy score is  0.8199611147116008
Precision score is  0.3344982526210684
F1 score is  0.4992548435171386
Recall score is   0.9838472834067548

Threshold 0.4:
Accuracy score is  0.8228127025275438
Precision score is  0.3344982526210684
F1 score is  0.4992548435171386
Recall score is   0.9838472834067548

Threshold 0.5:
Accuracy score is  0.8257939079714841
Precision score is  0.3344982526210684
F1 score is  0.4992548435171386
Recall score is   0.9838472834067548

Threshold 0.6:
Accuracy score is  0.8300712896953986
Precision score is  0.3344982526210684
F1 score is  0.4992548435171386
Recall score is   0.9838472834067548

Threshold 0.7:
Accuracy score is  0.8369410239792612
Precision score is  0.3344982526210684
F1 score is  0.4992548435171386
Recall score is   0.9838472834067548

Threshold 0.8:
Accuracy score is  0.8536616979909267
Precision score is  0.3344982526210684
F1 score is  0.4992548435171386
Recall score is   0.9838472834067548



In [ ]:
print(yprobab[:20])

[4.0644500e-02 5.8257421e-05 5.0719776e-05 7.1879203e-06 6.8201498e-06
 3.7571933e-06 4.8682418e-06 8.9085454e-01 1.8321476e-05 2.5973997e-05
 7.8184603e-06 9.4697505e-01 9.9733365e-01 6.7647841e-07 2.6502258e-09
 2.3121621e-02 9.6328613e-06 9.3133634e-01 1.5997452e-05 9.5125175e-01]
